In [5]:
import warnings
warnings.filterwarnings('ignore')
import os, re, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

from numba import njit, prange
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')

# Data 
DATA_START = '2000-01-01'
END_DATE = '2026-01-01'
BENCHMARK = '^SP500TR'
RF_TICKER = '^IRX' # 13-week T-bill, annualized percent
TRADING_DAYS = 252

# Rolling windows
FORMATION_MONTHS = 12 # form pairs on the first 12 months
TRADING_MONTHS = 6 # trade them on the next 6 months
STEP_MONTHS = 6 # then step forward 6 months
FIRST_FORMATION = '2001-01-01'
LAST_FORMATION = '2025-12-31'

# Pair selection 
K_OPTION = [2,4,7] # candidate cluster amounts, chosen by CVI for each window (Silhouette is the tie-breaker)
N_PAIRS = 20 # pairs traded per method

# Trading rule: following the Gatev et al. paper (2006)
ENTRY_SD = 2.0   # Open when the spread moves 2 formation-period standard deviations
LAG_DAYS = 1    # Trade the day after the signal
COST_LEVELS_BPS = [0, 5, 10, 20]

# Significance testing
N_BOOT, BLOCK_DAYS, BOOT_SEED = 5000, 20, 42

# Models 
METHODS = ['MAN', 'DTW', 'OCP', 'GATEV']
LABELS = {'MAN': 'Clustered Manhattan', 'DTW': 'Clustered DTW', 'OCP': 'Clustered OCP',
          'GATEV': 'Gatev Distance', 'Naive': 'Naive Buy-and-Hold'}
COLORS = {'MAN': '#9467bd', 'DTW': '#1f77b4', 'OCP': '#ff7f0e', 'GATEV': '#2ca02c', 'Naive': '#7f7f7f'}
PLANNED_COMPARISONS = [('MAN', 'GATEV'), ('DTW', 'MAN'), ('OCP', 'DTW')]

# Files
TAG = 'rolling_sp500'
CACHE_DIR = f'cache_{TAG}'
os.makedirs(CACHE_DIR, exist_ok=True)

print(f'Rolling design: form {FORMATION_MONTHS} months, trade {TRADING_MONTHS} months, step {STEP_MONTHS} months')
print(f'Models: {[LABELS[m] for m in METHODS]}')
print(f'Planned comparisons: {[f"{LABELS[a]} vs {LABELS[b]}" for a, b in PLANNED_COMPARISONS]}')
print(f'Cache folder: {CACHE_DIR}')

Rolling design: form 12 months, trade 6 months, step 6 months
Models: ['Clustered Manhattan', 'Clustered DTW', 'Clustered OCP', 'Gatev Distance']
Planned comparisons: ['Clustered Manhattan vs Gatev Distance', 'Clustered DTW vs Clustered Manhattan', 'Clustered OCP vs Clustered DTW']
Cache folder: cache_rolling_sp500


In [ ]:
hist_url = ('https://raw.githubusercontent.com/fja05680/sp500/master/'
            'S%26P%20500%20Historical%20Components%20%26%20Changes.csv')
hist = pd.read_csv(hist_url)
hist['date'] = pd.to_datetime(hist['date'])
hist = hist.sort_values('date').reset_index(drop=True)
hist.to_csv(f'sp500_membership_{TAG}.csv', index=False)

suffix_pattern = re.compile(r'-\d{6}$')
def clean(t):
    return suffix_pattern.sub('', t.strip()).replace('